# Network monitoring and traffic analysis

# Coding problem, 3.5.b: Traffic-anomaly detector

**Skill:** Detect Attacks · **Maps to:** 3.5.B · **Language:** Python 3 (pytest)

## Problem
Implement a simplified network traffic-anomaly detector. You are given a per-host **baseline**
(the normal profile for that host) and a window of **observed flow records**. Flag every host that
either exceeds its baseline byte-volume threshold or uses a destination port not in its baseline's
allowed-port set.

A **baseline** is a dict mapping host IP (str) to a dict with:
- `"max_bytes"`, integer, the maximum total outbound bytes this host normally sends in the window
- `"allowed_ports"`, list of integers, destination ports this host normally uses

An **observed flow record** is a dict with:
- `"host"`, source host IP string
- `"dst_port"`, destination port integer
- `"bytes"`, integer, bytes sent in this flow

A host may appear in multiple flow records within the same window; volumes accumulate.

## Function contract
Implement `solve(baseline, flows)` in the code editor (see the Reference section below).

- `baseline`, dict as described above
- `flows`, list of flow record dicts (any order)

Return a sorted `list[str]` of host IPs flagged as anomalous. A host is flagged if EITHER:
1. Its **total** bytes across all its flow records in this window exceeds `max_bytes`, OR
2. Any single flow record for that host uses a `dst_port` not present in `allowed_ports`.

A host not present in `baseline` at all is always flagged (unknown host, no baseline to compare
against, treated as an anomaly).

## Examples
```python
baseline = {
    "10.0.1.5": {"max_bytes": 500_000_000, "allowed_ports": [445, 139]},
    "10.0.2.9": {"max_bytes": 5_000_000, "allowed_ports": [80, 443]},
}
flows = [
    {"host": "10.0.1.5", "dst_port": 445, "bytes": 100_000_000},
    {"host": "10.0.1.5", "dst_port": 445, "bytes": 50_000_000},
    {"host": "10.0.2.9", "dst_port": 4444, "bytes": 1_000},   # unusual port -> flagged
]
solve(baseline, flows) == ["10.0.2.9"]
```

```python
baseline = {"10.0.3.1": {"max_bytes": 1_000_000, "allowed_ports": [443]}}
flows = [
    {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
    {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},   # total 1.8M > 1M -> flagged
]
solve(baseline, flows) == ["10.0.3.1"]
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).
- Result is sorted, contains no duplicate host IPs.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(baseline: dict, flows: list[dict]) -> list[str]
```

### Input

#### `baseline`, dict mapping host IP (str) -> profile dict
Each profile dict has:
- `"max_bytes"` (int): maximum normal total outbound bytes for this host in the window
- `"allowed_ports"` (list[int]): destination ports normally used by this host

#### `flows`, list of observed flow record dicts (any order, possibly multiple per host)
Each flow dict has:
- `"host"` (str): source host IP
- `"dst_port"` (int): destination port
- `"bytes"` (int): bytes sent in this flow

### Output (`result`)
Sorted `list[str]` of host IPs flagged as anomalous, no duplicates.

### Flagging rules
A host is flagged if ANY of the following hold:
1. Host is not a key in `baseline` (unknown host, no profile to compare against).
2. The sum of `"bytes"` across all of that host's flow records exceeds `baseline[host]["max_bytes"]`.
3. Any single flow record for that host has a `"dst_port"` not in `baseline[host]["allowed_ports"]`.

### Guarantees / rules
- Pure function; deterministic; no I/O; no network access.
- `baseline` and `flows` are not mutated by `solve`.
- Standard library only.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 3.5.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented.
"""
from typing import Dict, List


def solve(baseline: Dict, flows: List[Dict]) -> List[str]:
    """Flag hosts whose observed traffic deviates from their baseline profile.

    Args:
        baseline: host IP -> {"max_bytes": int, "allowed_ports": list[int]}
        flows:    list of {"host": str, "dst_port": int, "bytes": int}

    Returns:
        Sorted list of anomalous host IPs (no duplicates).
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def test_port_anomaly_flags_host():
    baseline = {
        "10.0.1.5": {"max_bytes": 500_000_000, "allowed_ports": [445, 139]},
        "10.0.2.9": {"max_bytes": 5_000_000, "allowed_ports": [80, 443]},
    }
    flows = [
        {"host": "10.0.1.5", "dst_port": 445, "bytes": 100_000_000},
        {"host": "10.0.1.5", "dst_port": 445, "bytes": 50_000_000},
        {"host": "10.0.2.9", "dst_port": 4444, "bytes": 1_000},
    ]
    assert impl.solve(baseline, flows) == ["10.0.2.9"]


def test_volume_anomaly_flags_host():
    baseline = {"10.0.3.1": {"max_bytes": 1_000_000, "allowed_ports": [443]}}
    flows = [
        {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
        {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
    ]
    assert impl.solve(baseline, flows) == ["10.0.3.1"]


def test_normal_traffic_not_flagged():
    baseline = {"10.0.5.5": {"max_bytes": 1_000_000, "allowed_ports": [80, 443]}}
    flows = [
        {"host": "10.0.5.5", "dst_port": 443, "bytes": 400_000},
        {"host": "10.0.5.5", "dst_port": 80, "bytes": 200_000},
    ]
    assert impl.solve(baseline, flows) == []


def test_unknown_host_flagged():
    baseline = {"10.0.1.1": {"max_bytes": 1_000_000, "allowed_ports": [80]}}
    flows = [
        {"host": "10.0.1.1", "dst_port": 80, "bytes": 100},
        {"host": "192.168.9.9", "dst_port": 80, "bytes": 100},
    ]
    assert impl.solve(baseline, flows) == ["192.168.9.9"]


def test_exact_boundary_not_flagged():
    baseline = {"10.0.6.1": {"max_bytes": 1_000, "allowed_ports": [80]}}
    flows = [{"host": "10.0.6.1", "dst_port": 80, "bytes": 1_000}]
    # Exactly at max_bytes is NOT an anomaly (only strictly exceeding is).
    assert impl.solve(baseline, flows) == []


def test_multiple_hosts_mixed_results():
    baseline = {
        "a": {"max_bytes": 100, "allowed_ports": [22]},
        "b": {"max_bytes": 100, "allowed_ports": [22]},
        "c": {"max_bytes": 100, "allowed_ports": [22]},
    }
    flows = [
        {"host": "a", "dst_port": 22, "bytes": 50},       # ok
        {"host": "b", "dst_port": 22, "bytes": 150},      # volume anomaly
        {"host": "c", "dst_port": 9999, "bytes": 10},     # port anomaly
    ]
    assert impl.solve(baseline, flows) == ["b", "c"]


def test_empty_flows():
    baseline = {"a": {"max_bytes": 100, "allowed_ports": [22]}}
    assert impl.solve(baseline, []) == []


def test_result_sorted_and_deduplicated():
    baseline = {"z": {"max_bytes": 10, "allowed_ports": [80]}, "a": {"max_bytes": 10, "allowed_ports": [80]}}
    flows = [
        {"host": "z", "dst_port": 999, "bytes": 1},
        {"host": "z", "dst_port": 999, "bytes": 1},
        {"host": "a", "dst_port": 999, "bytes": 1},
    ]
    assert impl.solve(baseline, flows) == ["a", "z"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
